In [1]:
import pandas as pd

In [1]:
pip list

Package                   Version
------------------------- -----------
annotated-types           0.7.0
anyio                     4.13.0
appnope                   0.1.4
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.4.0
certifi                   2026.5.20
cffi                      2.0.0
charset-normalizer        3.4.7
comm                      0.2.3
cryptography              48.0.0
debugpy                   1.8.21
decorator                 5.3.1
defusedxml                0.7.1
distro                    1.9.0
executing                 2.2.1
fastjsonschema            2.21.2
fqdn                      1.5.1
google-auth               2.53.0
google-genai              2.8.0
h11                       0.16.0
httpcore                  1.0.9
httpx            

In [3]:
!python --version

Python 3.14.4


## Learnings

Okay I'm in the correct env.

### What is RAG?
RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly.

In [2]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_KEY"))

In [3]:
# Building the document DB and the retriever
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [4]:
# Basically a list of the courses and path to the faq json data
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [5]:
# Since this is a local project, perhaps I'll save the dataset into a parquet file, I can get some compression from the
# repeated maybe course name or something like that
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [6]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [7]:
# saving documents to parquet file
documents_df = pd.DataFrame(documents)
documents_df.sort_values(by="course", inplace=True)

In [11]:
documents_df['id'] = documents_df['id'].astype(str)

In [ ]:
documents_df.info()

<class 'pandas.DataFrame'>
Index: 1208 entries, 551 to 1207
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        1208 non-null   str  
 1   course    1208 non-null   str  
 2   section   1208 non-null   str  
 3   question  1208 non-null   str  
 4   answer    1208 non-null   str  
dtypes: str(5)
memory usage: 900.3 KB


In [116]:
documents_df['full_content'] = documents_df['course'] + " " + documents_df['question'] + " " + documents_df['section'] + " " + documents_df['answer']

In [117]:
documents_df.to_parquet("./courses_faq.parquet", index=False)

In [118]:
documents_df = pd.read_parquet("./courses_faq.parquet")

In [119]:
# a light weight text search engine, not vector search, but more of a keyword search
from minsearch import Index
import json

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(json.loads(documents_df.to_json(orient="records")))

In [102]:
question = "What is the duration of the course?"
results = index.search(question, num_results=5)

# the index object during search has weight options and a filter dict option to further limit results bases on specific fields 
# in the data

# Trying filter options
results_filtered = index.search(question, num_results=5,
                                boost_dict={"answer": 2.0,"section": 0.5},
                                filter_dict={"course": "llm-zoomcamp"})

In [120]:
results_filtered

[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in t

In [121]:
results_filtered

[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in t

### Comparing lemmatized TFIDF + Cosine Similarity with this minisearch approach

In [48]:
documents_df

,id,course,section,question,answer,full_content
551,9e508f2212,data-engineering-zoomcamp,General Course-Related Questions,Course: When does the course start?,A new cohort runs roughly January–April every ...,data-engineering-zoomcamp Course: When does th...
552,bfafa427b3,data-engineering-zoomcamp,General Course-Related Questions,Course: What are the prerequisites for this co...,"To get the most out of this course, you should...",data-engineering-zoomcamp Course: What are the...
553,3f1424af17,data-engineering-zoomcamp,General Course-Related Questions,Course: Can I still join the course after the ...,"Yes, even if you don't register, you're still ...",data-engineering-zoomcamp Course: Can I still ...
554,52217fc51b,data-engineering-zoomcamp,General Course-Related Questions,Course: I have registered for the Data Enginee...,You don't need a confirmation email. You're ac...,data-engineering-zoomcamp Course: I have regis...
555,33fc260cd8,data-engineering-zoomcamp,General Course-Related Questions,Course: What can I do before the course starts?,Get the basic environment ready ahead of time:...,data-engineering-zoomcamp Course: What can I d...
...,...,...,...,...,...,...
1203,716ff05962,mlops-zoomcamp,Capstone Project,pytest doesn't recognize my installed librarie...,This usually happens because VS Code is using ...,mlops-zoomcamp pytest doesn't recognize my ins...
1204,0a7c21fee9,mlops-zoomcamp,Capstone Project,Is it a group project?,"No, the capstone is a solo project.",mlops-zoomcamp Is it a group project? Capstone...
1205,9ae63755b8,mlops-zoomcamp,Capstone Project,"Do we submit 2 projects, what does attempt 1 a...",You only need to submit one project. If the su...,"mlops-zoomcamp Do we submit 2 projects, what d..."
1206,b6c7eb17d8,mlops-zoomcamp,Capstone Project,How is my capstone project going to be evaluated?,Each submitted project will be evaluated by th...,mlops-zoomcamp How is my capstone project goin...


In [54]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/pithun/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pithun/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /Users/pithun/nltk_data...


True

In [ ]:

from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from wordcloud import STOPWORDS
from emot.emo_unicode import UNICODE_EMOJI, EMOTICONS_EMO
import dill
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# The code below basically adds escape characters  \\ to the emoticon characters since regex is used to
# remove them.
# for example, we have :-) but ) is a group character in regex so, we had to escape it
new_string = ''
new_emoticons = []
meanings = []
to_escape = [']', '[', '+', '*', '(', ')', '?', '{', '}', '|', '.']
for a, mean in EMOTICONS_EMO.items():
    meanings.append(mean)
    #print(a)
    for each in a:
        if each in to_escape:
            new_string +=  '\\'+each
        else:
            new_string+=each
    new_emoticons.append(new_string)
    new_string = ''

# the problem here was that (\\\\) doesn't escape the second parenthesis, so i had to change it to \\)
for ind, new in enumerate(new_emoticons):
    new_emoticons[ind] = new.replace('(\\\\)', '(\\)')

stopwords_better = list(STOPWORDS)
for remove in ['\'', '^', '^^', 'doe', 'ha', 'wa', '\'i', '\'the', '\'s', '\'cause', '\'em']:
    stopwords_better.append(remove)

def emoji_to_text(text_):
    '''This function replaces emojis with their meanings'''
    for emoji, meaning in UNICODE_EMOJI.items():
        text_ = text_.replace(emoji, meaning)
    return text_

def convert_emoticons(text_):
    ''' This function replaces emoticons which are basically symbols to represent expressions with their meaning
    some are in the contents so it makes sense to remove them.'''
    for emoticon, meaning in zip(new_emoticons, meanings):
      #basically emoticon becomes a pattern here that's why it's important to escape those characters
        text_ = re.sub(emoticon, meaning, text_)
    return text_

def remlinks_symbs(df):
    '''This function creates a new column with the links, mentions, symbols, numbers of more than 1character
    removed and emojis, emoticons and replaced with their meaning in the data set'''
    df=df.copy()
    df['content'] = df.full_content

    # replacing link with space
    # we noticed some links don't have :// it's just https:youtu... and some start from www.
    # example www.youtube.com/watch?v=2o5FHtUaKNQ so we had to tweak the regex multiple times to meet the
    # needs.
    df.content.replace(r'(https?:/?/?|ftp:/?/?|www|[yY]outube)+?[\w\-\?\=\%.]+?.[\w\!\£\$\%\^\&\*\(\)\_\+\-\=\{\\}\~\[\]\#\:\@\;\'\<\>\?\,\.\/\\]+', ' ', regex= True, inplace= True)

    # replacing mentions with space
    df.content.replace(r'@[\w\!\£\$\%\^\&\*\(\)\_\+\-\=\{\\}\~\[\]\#\:\@\;\'\<\>\?\,\.\/\\]+', ' ',
                    regex= True, inplace= True)

    #reducing all to lowercase
    df['content'] = df['content'].str.lower()

    # Replacing emoticons with their meaning
    df.content = df.content.apply(convert_emoticons)

    #replacing emojis with their meanings
    df.content = df.content.apply(emoji_to_text)

    # replacing one or more symbols except ', $, % and _ with space
    df.content.replace(r'[^\w$%_\']+', ' ', regex = True, inplace = True)

    #removes numbers longer than two since 4w3 has a meaning removing numbers will leave just w
    #df.content.replace(r'\d\d+', '', regex = True, inplace = True)

    #replacing underscores with nothing
    #df.content.replace(r'_', '', regex = True, inplace = True)
    return df

def lemmatized(sentence):
     # Lemmatizer function using parts of speech tagging.
    wnl = WordNetLemmatizer()
    lemmatized = []
    for word, tag in pos_tag(sentence.split()):
        if tag.startswith("NN"):
            #print(word, tag)
            lemmatized.append(wnl.lemmatize(word, pos='n'))
        elif tag.startswith('VB'):
            #print(word, tag)
            lemmatized.append(wnl.lemmatize(word, pos='v'))
        elif tag.startswith('JJ'):
            #print(word, tag)
            lemmatized.append(wnl.lemmatize(word, pos='a'))
        else:
            lemmatized.append(word)
    return lemmatized

def BOW(train_data, ngrams=1, max_feat=1500):
    """
    Creates TF-IDF vectors and keeps a lookup table
    containing original and cleaned text.
    """

    train_data = remlinks_symbs(train_data.copy())

    if ngrams == 1:

        vectorizer = TfidfVectorizer(
            max_features=max_feat,
            stop_words=stopwords_better,
            tokenizer=lemmatized
        )

        tfidf_matrix = vectorizer.fit_transform(train_data.content)

        with open(f"vectorizers/tfidf_vectorizer_{max_feat}.dill", "wb") as f:
            dill.dump(vectorizer, f)

        lookup_df = train_data[
            ["full_content", "content"]
        ].reset_index(drop=True)

        return tfidf_matrix, lookup_df

    raise NotImplementedError("Only unigram supported for now")

def BOW_old(train_data, ngrams=1, max_feat= 1500, split_data = False):
    '''This does all cleaning and returns the bag of documents vector for both train and test data
    note that you have to manually supply train and test data'''

    # for train_data
    # here we apply the clean function above
    train_data = remlinks_symbs(train_data)

    #for test_data
    #test_data = remlinks_symbs(test_data)

    if ngrams == 1:
        #train_data.content.replace(r'[^\w^\']+', ' ', regex = True, inplace = True)
        #train_data.content = train_data.content.apply(rem_pos)

        #test_data.content.replace(r'[^\w^\']+', ' ', regex = True, inplace = True)
        #test_data.content = test_data.content.apply(rem_pos)

        # tokenizing words and making the vectors
        vectorizer=TfidfVectorizer(max_features= max_feat, stop_words= stopwords_better, tokenizer=
                                   lemmatized) #, token_pattern= '(?u)\b\\w+\'\w+\b')
        # the reason it tokenizes "'cause" is because ' is recognized as a boundary

        char_array_tr = vectorizer.fit_transform(train_data.content).toarray()
        
        with open(f"vectorizers/tfidf_vectorizer_{max_feat}.dill", "wb") as f:
            dill.dump(vectorizer, f)
        #dump(vectorizer, f"models/tfidf_vectorizer_{max_feat}.joblib")
        #char_array_te = vectorizer.transform(test_data.content).toarray()

        # Turning into dataframes
        frequency_matrix_tr = pd.DataFrame(char_array_tr, columns= vectorizer.get_feature_names_out())
        #frequency_matrix_te = pd.DataFrame(char_array_te, columns= vectorizer.get_feature_names_out())
    else:
        pass
    if split_data== True:
        return (frequency_matrix_tr, frequency_matrix_te)
    else:
        return frequency_matrix_tr

def get_test_vector(test_data, vectorizer_path):

    test_data = remlinks_symbs(test_data.copy())

    with open(vectorizer_path, "rb") as f:
        vectorizer = dill.load(f)

    return vectorizer.transform(test_data.content)


def cosine_index(
        search_vector,
        train_vectors,
        lookup_df,
        n=5):

    similarities = cosine_similarity(
        search_vector,
        train_vectors
    )[0]

    top_indices = np.argsort(similarities)[::-1][:n]

    results = lookup_df.iloc[top_indices].copy()

    results["cosine_similarity"] = similarities[top_indices]

    return results[
        [
            "full_content",
            "content",
            "cosine_similarity"
        ]
    ]

def cos_search(query, vectorizer_path, train_vectors, lookup_df, n=5):
    query_df = pd.DataFrame({"full_content": [query]})
    query_vector = get_test_vector(query_df, vectorizer_path)
    results = cosine_index(
        query_vector,
        train_vectors,
        lookup_df,
        n=n
    )
    return results


In [82]:
# Testing the lemmatizer function
lemmatized("the churches are beautiful good")

['the', 'church', 'be', 'beautiful', 'good']

In [141]:
query_df = pd.DataFrame({
    "full_content": [
        "Why are we not using Langchain in the llm-zoomcamp course ?"
    ]
})

query_vector = get_test_vector(
    query_df,
    "vectorizers/tfidf_vectorizer_50.dill"
)

results = cosine_index(
    query_vector,
    train_vectors,
    lookup_df,
    n=10
)

print(results['full_content'])

447     llm-zoomcamp Why was .dot(...) used directly t...
343     data-engineering-zoomcamp What is the use of R...
450     llm-zoomcamp Why does cosine similarity reduce...
29      data-engineering-zoomcamp Why does the course ...
1178    mlops-zoomcamp Is it mandatory to use a refere...
968     mlops-zoomcamp Homework 3, 2025 Cohort - Do I ...
587     machine-learning-zoomcamp Standard Deviation D...
822     machine-learning-zoomcamp Can we use PyTorch f...
414     llm-zoomcamp Why are we not using Langchain in...
671     machine-learning-zoomcamp Multiple thresholds ...
Name: full_content, dtype: str


/var/folders/zw/mv93vl_x2f36wm65g8cfg6t40000gn/T/ipykernel_10971/897206308.py:60: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df.content.replace(r'(https?:/?/?|ftp:/?/?|www|[yY]outube)+?[\w\-\?\=\%.]+?.[\w\!\£\$\%\^\&\*\(\)\_\+\-\=\{\\}\~\[\]\#\:\@\;\'\<\>\?\,\.\/\\]+', ' ', regex= True, inplace= True)
/var/folders/zw/mv93vl_x2f36w

### Text Search Functions

In [106]:
# creating search function
def search(question, course="llm-zoomcamp", num_results=5, boost_dict={"question": 2.0, "section": 0.5}):
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

In [39]:
# functions to build context from index search result.
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

# building prompt for LLM with user question and context from search results
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        # in place text replacement
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
#### Quick comparison: Text Vs Vector Search
query="what do I need to do to get a certificate for the llm-zoomcamp course?"
res=search(query)
cos_res=cos_search(query, "vectorizers/tfidf_vectorizer_300.dill", train_vectors, lookup_df, n=5):

In [35]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [42]:
question = "What is the duration of the course?"
search_results = search(question, num_results=3)

prompt = build_prompt(question, search_results)

print(prompt)

Question:
What is the duration of the course?

Context:
General Course-Related Questions
Q: What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
A: The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram & Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).

Don’t post questions in chat as they may be missed if the room is very active.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your 

In [43]:
# Passing to the LLM
from google import genai
from google.genai import types

def get_llm_response(question):
    search_res=search(question, num_results=3)
    prompt=build_prompt(question, search_res)

    response = client.models.generate_content(
            model="gemini-2.5-flash",
            config=types.GenerateContentConfig(
                system_instruction=INSTRUCTIONS),
            contents=prompt)
    
    return response.text

In [44]:
worried_about_graduation_criteria="What do I need to do to graduate from the course?"

llm_answer = get_llm_response(worried_about_graduation_criteria)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}